In [ ]:
!pip install google_play_scraper
!pip install textblob
from google_play_scraper import app
import pandas as pd
import numpy as np
import sklearn
import requests
import matplotlib.pyplot as plt
import matplotlib.dates as dates
import seaborn as sns
import textblob
#from wordcloud import WordCloud
from pathlib import Path
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix,classification_report, accuracy_score

import pickle
import re
import time
import datetime                              # access to %%time, for timing individual notebook cells
import os
from PIL import Image
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

%matplotlib inline
%config InlineBackend.figure_format='retina'

# plt.style.use('seaborn')
plt.rcParams["figure.figsize"] = (15,10)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.8 MB/s eta 0:00:00


In [ ]:

from google_play_scraper import app, Sort, reviews_all

nhs_reviews = reviews_all(
    'com.pure.indosat.care',
    sleep_milliseconds=0, # defaults to 0
    lang='id', # indonesia version
    sort=Sort.NEWEST, # defaults to Sort.MOST_RELEVANT
)

In [ ]:
#Save the NHS Apps reviews into dataframe
df_nhsrev = pd.DataFrame(np.array(nhs_reviews),columns=['content'])
df_nhsrev = df_nhsrev.join(pd.DataFrame(df_nhsrev.pop('content').tolist()))
df_nhsrev.to_csv(r'C:\Users\alqis\Documents\crqwling.csv', index=False)
df_nhsrev.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,1cdfea89-0393-484b-a223-606ad79e7792,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,bagus untuk di coba,5,0,82.13.0,2026-04-17 05:39:19,None,NaT,82.13.0
1,dd85a1bd-ff68-455e-b8e3-55187b18dd6d,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,aplikasi ini sangat membantu dn kuota ny pun m...,5,0,None,2026-04-17 05:38:08,None,NaT,None
2,ab2e1a00-e954-4416-90fa-de5f112c03a1,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"aplikasi ga guna, pulsa saya cukup tapi selalu...",1,0,82.13.1,2026-04-17 05:23:08,"Selamat sore Bapak Syaiful, maaf sebelumnya. S...",2026-04-17 09:08:05,82.13.1
3,de8ff7ae-0d8f-450f-a5cb-65ec879e256e,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,ga rekomendasi,1,0,None,2026-04-17 05:16:45,"Selamat sore Pelanggan IM3, mohon maaf atas pe...",2026-04-17 08:52:16,None
4,01c96b88-97e5-4b80-88ae-f669a95adfa2,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,bagus,5,0,82.13.1,2026-04-17 05:00:55,None,NaT,82.13.1


In [8]:
# labeling sederhana berdasarkan rating dari pengguna
def label_sentiment(score):
    if score <= 2:
        return 'Negative'
    elif score == 3:
        return 'Neutral'
    else:
        return 'Positive'

df_nhsrev['sentiment'] = df_nhsrev['score'].apply(label_sentiment)
negatif = df_nhsrev[df_nhsrev['sentiment'] == 'Negative'].sample(n=50)
netral  = df_nhsrev[df_nhsrev['sentiment'] == 'Neutral'].sample(n=50)
positif = df_nhsrev[df_nhsrev['sentiment'] == 'Positive'].sample(n=50)

df_sample = pd.concat([negatif, netral, positif])
df_sample.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,sentiment
1671243,dc19abbd-a6aa-4daf-ab4b-3d2607c95212,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,error terus,1,1,80.2.2,2020-08-20 13:34:49,"Hai kak Hasyi, maaf atas ketidaknyamanannya. K...",2020-08-25 09:17:54,80.2.2,Negative
2071558,e0d1f352-da5f-4dca-a3b1-5672b4aac612,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"Tolong di perbaiki koneksi nya sangat butuk, u...",1,105,80.0.3,2020-03-11 18:48:40,"Hai Explor Gamming, kami akan terus tingkatkan...",2020-03-12 06:37:53,80.0.3,Negative
861213,834b628f-ff43-42c1-b244-2c3b5c4ca9fb,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Saya di undang teman via wa trus saya klik lin...,1,0,None,2021-12-02 17:36:09,"\nSore Kak Geshelia, maaf atas ketidaknyamanan...",2021-12-04 08:58:21,None,Negative
781987,de81efff-7a63-4e58-98e8-7854aaf27343,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Najisss dikasih ujan gerimis aja bukan ngeleg ...,1,0,None,2022-02-04 13:37:50,"\nHai kak, maaf atas ketidaknyamanannya. Terka...",2022-02-06 03:36:05,None,Negative
243886,394a5fbb-a989-4ea2-b558-358812c0dab2,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,udah di update malah gabisa di buka,1,0,82.1.0,2024-03-13 12:29:36,"Hi Kak Deny, maaf sebelumnya. Pastikan sudah m...",2024-03-13 15:38:56,82.1.0,Negative
